In [2]:
#Package Conflicts

# !pip uninstall -y transformers huggingface_hub
# !pip cache purge
# !pip install "transformers==4.56.1" "huggingface_hub==0.35.0"
# !pip install -U sentence-transformers==5.1.0

# !pip show torch

# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# !pip install hf_xet

In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

In [4]:
def _encode_longformer(texts):
    """Encode using Longformer with mean pooling"""
    embeddings = []
    
    for text in texts:
        # Tokenize
        inputs = tokenizer(text, return_tensors='pt', 
                                truncation=True, max_length=4096, 
                                padding=True).to(device)
    
        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            # Mean pooling over tokens
            embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings.append(embedding[0])
    
    return np.array(embeddings)

In [5]:
def encode(texts):
    """Encode texts into embeddings"""
    if model_type == 'sbert':
        return model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    elif model_type == 'longformer':
        return _encode_longformer(texts)

In [6]:
def predict_triple(anchor, text_a, text_b):
    """
    Predict which text is more similar to anchor
    
    Returns:
        True if text_a is closer, False if text_b is closer
    """
    # Encode all three texts
    embeddings = encode([anchor, text_a, text_b])
    
    # Calculate cosine similarities
    sim_a = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    sim_b = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]
    
    return sim_a > sim_b

In [7]:
def evaluate(df):
    """
    Evaluate on dataset
    
    Args:
        df: DataFrame with columns ['anchor_text', 'text_a', 'text_b', 'text_a_is_closer']
    
    Returns:
        accuracy, predictions, similarities
    """
    predictions = []
    similarities_a = []
    similarities_b = []
    
    print(f"Evaluating {len(df)} samples...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Encode
        embeddings = encode([row['anchor_text'], row['text_a'], row['text_b']])
        
        # Calculate similarities
        sim_a = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
        sim_b = cosine_similarity([embeddings[0]], [embeddings[2]])[0][0]
        
        # Predict
        pred = sim_a > sim_b
        predictions.append(pred)
        similarities_a.append(sim_a)
        similarities_b.append(sim_b)
    
    # Calculate accuracy
    df['prediction'] = predictions
    df['sim_a'] = similarities_a
    df['sim_b'] = similarities_b
    accuracy = (df['prediction'] == df['text_a_is_closer']).mean()
    
    return accuracy, df

In [9]:
# df = pd.read_json('./Data/SemEval2026-Task_4-sample-v1/sample_track_a.jsonl', lines=True)
df = pd.read_json('../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl', lines=True)
df.head()


,anchor_text,text_a,text_b,text_a_is_closer
0,The book follows an international organization...,The old grandmother Tina arrives in town to at...,The nano-plague that poisoned Earth's water su...,False
1,"Glenn Tyler (Elvis Presley), a childish 25-yea...","Bill Babbitt supported the death penalty, unti...",A white-collar suburban father Kyle (Fran Kran...,True
2,Signaller Charles Plumpick (Bates) is a kilt-w...,"Sid, Russ and Jerry are three wannabe criminal...",Brendan Byers III is a rich playboy who enlist...,False
3,Barbara is married to the distinguished profes...,Eddie Quinn's unruly wife Maureen drinks and s...,Jerome Littlefield is an orderly at a hospital...,False
4,A wealthy widower locks up his two grown-up ch...,Barbara is married to the distinguished profes...,Stefano (Lino Capolicchio) arrives in a villag...,False


In [10]:
print(f"Loaded {len(df)} samples")
print(f"Columns: {df.columns.tolist()}")

Loaded 200 samples
Columns: ['anchor_text', 'text_a', 'text_b', 'text_a_is_closer']


In [11]:
models_to_test = [
    ('sentence-transformers/all-MiniLM-L6-v2', 'sbert'),
    ('sentence-transformers/all-mpnet-base-v2', 'sbert'),
    ('allenai/longformer-base-4096', 'longformer'),
]
results =[]
for mn,mt in models_to_test:
    model_name=mn
    model_type=mt

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Loading {model_name} on {device}...")

    if model_type == 'sbert':
        model = SentenceTransformer(model_name, device=device)
    elif model_type == 'longformer':
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).to(device)

    accuracy, predictions_df = evaluate(df.copy())


    # Store results
    results.append({
        'model': model_name,
        'type': model_type,
        'accuracy': accuracy,
        'predictions_df': predictions_df
    })

    print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

    # Show some statistics
    correct = (predictions_df['prediction'] == predictions_df['text_a_is_closer']).sum()
    total = len(predictions_df)
    print(f"  Correct: {correct}/{total}")

    # Analyze errors
    errors = predictions_df[predictions_df['prediction'] != predictions_df['text_a_is_closer']]
    if len(errors) > 0:
        avg_sim_diff = (errors['sim_a'] - errors['sim_b']).abs().mean()
        print(f"  Error cases: {len(errors)}")

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")

summary_df = pd.DataFrame([
    {
        'Model': r['model'],
        'Type': r['type'],
        'Accuracy': f"{r['accuracy']:.4f}" if 'accuracy' in r else 'ERROR',
        'Percentage': f"{r['accuracy']*100:.2f}%" if 'accuracy' in r else 'ERROR'
    }
    for r in results
])

print(summary_df.to_string(index=False))

# Find best model
valid_results = [r for r in results if 'accuracy' in r and r['accuracy'] > 0]
if valid_results:
    best = max(valid_results, key=lambda x: x['accuracy'])
    print(f"\n Best Model: {best['model']}")
    print(f"   Accuracy: {best['accuracy']:.4f} ({best['accuracy']*100:.2f}%)")

Loading sentence-transformers/all-MiniLM-L6-v2 on cpu...
Evaluating 200 samples...


100%|██████████| 200/200 [00:17<00:00, 11.22it/s]



Accuracy: 0.5500 (55.00%)
  Correct: 110/200
  Error cases: 90
Loading sentence-transformers/all-mpnet-base-v2 on cpu...
Evaluating 200 samples...


100%|██████████| 200/200 [02:04<00:00,  1.60it/s]



Accuracy: 0.6150 (61.50%)
  Correct: 123/200
  Error cases: 77
Loading allenai/longformer-base-4096 on cpu...
Evaluating 200 samples...


100%|██████████| 200/200 [08:30<00:00,  2.55s/it]


Accuracy: 0.4650 (46.50%)
  Correct: 93/200
  Error cases: 107

SUMMARY
                                  Model       Type Accuracy Percentage
 sentence-transformers/all-MiniLM-L6-v2      sbert   0.5500     55.00%
sentence-transformers/all-mpnet-base-v2      sbert   0.6150     61.50%
           allenai/longformer-base-4096 longformer   0.4650     46.50%

 Best Model: sentence-transformers/all-mpnet-base-v2
   Accuracy: 0.6150 (61.50%)
